In [1]:
import oceanbench

oceanbench.__version__

'0.6.0'

### Open challenger datasets

> Insert here the code that opens the challenger dataset as `challenger_dataset: xarray.Dataset`

In [2]:
# Open HClimRep forecast sample with xarray
from datetime import datetime
import xarray

challenger_dataset: xarray.Dataset = xarray.open_mfdataset(
    [
        "https://s3.waw3-1.cloudferro.com/oceanbench-bucket/public/ml-forecast-outputs/hclimrep/20240103.zarr",
    ],
    engine="zarr",
    preprocess=lambda dataset: dataset.rename({"time": "lead_day_index"}).assign({"lead_day_index": range(10)}),
    combine="nested",
    concat_dim="first_day_datetime",
    parallel=True,
).assign(
    {
        "first_day_datetime": [
            datetime.fromisoformat("2024-01-03"),
        ]
    }
)
challenger_dataset


<xarray.Dataset> Size: 1GB
Dimensions:             (first_day_datetime: 1, lead_day_index: 10, depth: 9,
                         latitude: 672, longitude: 1440)
Coordinates:
  * depth               (depth) float64 72B 0.494 47.37 92.33 ... 453.9 541.1
  * latitude            (latitude) float64 5kB -78.0 -77.75 -77.5 ... 89.5 89.75
  * longitude           (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
  * lead_day_index      (lead_day_index) int64 80B 0 1 2 3 4 5 6 7 8 9
  * first_day_datetime  (first_day_datetime) datetime64[us] 8B 2024-01-03
Data variables:
    so                  (first_day_datetime, lead_day_index, depth, latitude, longitude) float32 348MB dask.array<chunksize=(1, 1, 1, 672, 1440), meta=np.ndarray>
    thetao              (first_day_datetime, lead_day_index, depth, latitude, longitude) float32 348MB dask.array<chunksize=(1, 1, 1, 672, 1440), meta=np.ndarray>
    uo                  (first_day_datetime, lead_day_index, depth, latitude, longitude) float32 348MB dask.array<chunksize=(1, 1, 1, 672, 1440), meta=np.ndarray>
    vo                  (first_day_datetime, lead_day_index, depth, latitude, longitude) float32 348MB dask.array<chunksize=(1, 1, 1, 672, 1440), meta=np.ndarray>
    zos                 (first_day_datetime, lead_day_index, latitude, longitude) float32 39MB dask.array<chunksize=(1, 1, 672, 1440), meta=np.ndarray>
Attributes:
    Conventions:                                        CF-1.8
    area:                                               Global
    challenger:                                         hclimrep
    institution:                                        ECMWF
    references:                                         https://huggingface.c...
    source:                                             WeatherGenerator infe...
    title:                                              WeatherGenerator GLOR...
    oceanbench_reference_depth_grid_rounding_decimals:  3
    oceanbench_reference_target_depths_m:               [0.494, 47.374, 92.32...

### Evaluation configuration

In [3]:
region = 'global'

### Evaluation of challenger dataset using OceanBench

#### Root Mean Square Deviation (RMSD) of variables compared to GLORYS reanalysis

In [4]:
oceanbench.metrics.rmsd_of_variables_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.055127,0.054777,0.053464,0.052383,0.054444,0.056013,0.055344,0.056701,0.057688,0.058754
Temperature (°C) [sea_water_potential_temperature]{surface},0.572562,0.589808,0.600123,0.615373,0.623304,0.639101,0.662769,0.676329,0.678897,0.686796
Salinity (PSU) [sea_water_salinity]{surface},0.427750,0.427327,0.423562,0.418770,0.414577,0.411940,0.409621,0.404535,0.401634,0.399622
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.113142,0.114368,0.115939,0.116954,0.117706,0.118746,0.120705,0.123222,0.123145,0.123233
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.114036,0.116545,0.118221,0.119320,0.121440,0.124006,0.126332,0.128695,0.129619,0.132532
Temperature (°C) [sea_water_potential_temperature]{50m},0.803085,0.792008,0.781851,0.772884,0.763567,0.760466,0.763847,0.773563,0.776398,0.783024
Salinity (PSU) [sea_water_salinity]{50m},0.203771,0.200880,0.199600,0.198652,0.197057,0.196390,0.196573,0.196595,0.195912,0.195513
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.105865,0.105821,0.105872,0.105659,0.105660,0.106358,0.107534,0.108275,0.108467,0.108549
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.106873,0.107006,0.107008,0.106898,0.106900,0.107909,0.108899,0.110239,0.110659,0.111068
Temperature (°C) [sea_water_potential_temperature]{100m},0.971146,0.961076,0.951141,0.942680,0.934157,0.930945,0.930270,0.933707,0.937397,0.946605


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLORYS reanalysis

In [5]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},37.235407,37.487684,38.254598,38.699293,39.502512,40.039692,40.669881,41.632248,42.077665,42.548158


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLORYS reanalysis

In [6]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.186760,0.178474,0.173793,0.169088,0.165174,0.163838,0.163106,0.161488,0.160586,0.159629
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.209102,0.203212,0.198511,0.192334,0.186562,0.182706,0.180033,0.180869,0.181624,0.178250


#### Root Mean Square Deviation (RMSD) of variables compared to observations

In [7]:
oceanbench.metrics.rmsd_of_variables_compared_to_observations(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10,Observations
Temperature (°C) [sea_water_potential_temperature]{surface},0.677227,0.858736,0.878209,0.836048,0.784352,0.817037,0.826742,0.835153,0.903500,0.861784,3035
Temperature (°C) [sea_water_potential_temperature]{0-5m},0.563329,0.582710,0.621782,0.598072,0.688436,0.637540,0.799841,0.699366,0.697390,0.744993,3811
Temperature (°C) [sea_water_potential_temperature]{5-100m},0.914800,1.042817,0.803699,0.919835,0.968897,0.919459,0.952523,0.935402,1.049621,1.141757,56445
Temperature (°C) [sea_water_potential_temperature]{100-300m},0.864658,0.965933,0.783452,0.906315,0.864228,0.826525,0.906661,0.993496,0.870077,0.983219,50949
Temperature (°C) [sea_water_potential_temperature]{300-600m},0.551706,0.565596,0.496348,0.530916,0.502269,0.479391,0.506443,0.540986,0.536357,0.535315,50756
Salinity (PSU) [sea_water_salinity]{0-5m},0.172638,0.200234,0.256796,0.339495,0.229922,0.198103,0.178093,0.215894,0.164847,0.221420,3349
Salinity (PSU) [sea_water_salinity]{5-100m},0.159032,0.250829,0.345222,0.255399,0.295600,0.334565,0.288937,0.209121,0.245847,0.369674,45248
Salinity (PSU) [sea_water_salinity]{100-300m},0.133136,0.146239,0.104975,0.131200,0.122071,0.109082,0.129969,0.118325,0.126061,0.145898,41916
Salinity (PSU) [sea_water_salinity]{300-600m},0.077217,0.089034,0.070940,0.081267,0.074077,0.067500,0.071362,0.079163,0.079553,0.092036,42973
Sea level anomaly (m) [sea_surface_height_above_geoid]{surface},0.058874,0.059684,0.061405,0.063417,0.065435,0.069286,0.068665,0.069338,0.070700,0.071924,302215


#### Deviation of Lagrangian trajectories compared to GLORYS reanalysis

In [8]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Lagrangian trajectory deviation (km) []{surface},9.799958,19.003403,27.852198,36.398422,44.826904,53.154781,61.263115,69.056549


#### Root Mean Square Deviation (RMSD) of variables compared to GLO12 analysis

In [9]:
oceanbench.metrics.rmsd_of_variables_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.041526,0.045029,0.047947,0.048938,0.054112,0.057574,0.059637,0.062286,0.063891,0.064877
Temperature (°C) [sea_water_potential_temperature]{surface},0.463515,0.477498,0.498493,0.522624,0.539854,0.567654,0.601259,0.627009,0.637977,0.654720
Salinity (PSU) [sea_water_salinity]{surface},0.377125,0.388018,0.398627,0.406797,0.417185,0.432519,0.443376,0.451738,0.460961,0.469999
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.074208,0.082729,0.091231,0.096659,0.102982,0.108355,0.115399,0.121871,0.124183,0.126750
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.076980,0.086787,0.094359,0.100907,0.107474,0.114486,0.120294,0.126527,0.129815,0.133122
Temperature (°C) [sea_water_potential_temperature]{50m},0.592853,0.610689,0.633188,0.656328,0.689149,0.720206,0.756729,0.787682,0.807284,0.819313
Salinity (PSU) [sea_water_salinity]{50m},0.154386,0.160028,0.163527,0.166879,0.171335,0.176751,0.181438,0.186444,0.189075,0.191092
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.066493,0.070323,0.074998,0.079518,0.084815,0.090689,0.097216,0.102370,0.105644,0.107217
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.068168,0.072231,0.076043,0.080862,0.085220,0.090478,0.096433,0.102374,0.105204,0.107537
Temperature (°C) [sea_water_potential_temperature]{100m},0.632155,0.647087,0.665020,0.686312,0.721349,0.754929,0.786483,0.823690,0.843264,0.858521


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLO12 analysis

In [10]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},36.733836,36.804332,37.346954,37.98067,38.588166,39.271397,40.124255,41.031648,41.512613,42.20763


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLO12 analysis

In [11]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.178118,0.171708,0.166910,0.164214,0.161216,0.161465,0.162339,0.163017,0.163743,0.162968
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.196876,0.191594,0.188639,0.185534,0.183242,0.181740,0.177165,0.180622,0.185726,0.180956


#### Deviation of Lagrangian trajectories compared to GLO12 analysis

In [12]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Lagrangian trajectory deviation (km) []{surface},6.493165,12.883559,19.382597,25.867849,32.503147,39.443001,46.514145,53.616814
